In [ ]:
from astropy import units
from astropy.time import Time
from astropy.visualization import quantity_support

from matplotlib import pyplot

import numpy 

from src.selflensing.fitting import self_lensing_system_from_fit
from src.selflensing.generators import LightcurveGenerator, RVGenerator
from src.selflensing.models import SelfLensingSystem

In [ ]:
quantity_support()

In [ ]:
LENSING_METHOD = 'IntegratedWittMao'

In [ ]:
sls = SelfLensingSystem(
    a=1 * units.AU,
    i=0.5*numpy.pi * units.rad,
    Macc=10.04*units.M_sun,
    Mcomp=1*units.M_sun,
    Rcomp=1*units.R_sun,
    e=0 * units.dimensionless_unscaled,
)
sls.porb, sls.tau_eff, sls.b, sls.b_max, sls.rE(), sls.i, sls.tau_eff

In [ ]:
start_time = Time("2025-01-01")
end_time = Time("2026-01-01")
peak_time = Time("2025-02-01")

In [ ]:
lcg = LightcurveGenerator(sls, start_time=start_time, end_time=end_time, peak_time=peak_time, ev=True, db=True, sl=True, lensing_method=LENSING_METHOD)
rvg = RVGenerator(sls, start_time=start_time, end_time=end_time, peak_time=peak_time)

In [ ]:
pyplot.scatter((lcg.time_offsets), lcg.flux_array)

In [ ]:
pyplot.scatter((rvg.time_offsets), rvg.rv)

In [ ]:
import numpy
peak_mag = numpy.max(lcg.timeseries['flux']) / numpy.median(lcg.timeseries['flux'])
min_mag = peak_mag * 0.999
max_mag = peak_mag * 1.001
min_mag, peak_mag, max_mag

In [ ]:
non_detection_times = [t for t in lcg.time_offsets if t not in lcg.peak_time_offsets]

In [ ]:
from src.selflensing.calc import estimate_porb

p = estimate_porb(sls.tau_eff, min_mag, max_mag, sls.Rcomp)
p

In [ ]:
from astropy import units
bin_starts = units.Quantity(p[0])
bin_ends = units.Quantity(p[1])
bin_counts = numpy.array(p[2])
total_count = sum(bin_counts)

bin_widths = bin_ends - bin_starts
bin_centres = (bin_starts + bin_ends) / 2
n_density = (bin_counts/total_count) / bin_widths.to(units.year).value

pyplot.plot(bin_centres.to(units.year).value, n_density)
pyplot.xscale('log')
pyplot.axvline(sls.porb.to(units.year).value)

In [ ]:
pyplot.plot(bin_centres.to(units.year).value, n_density)
pyplot.xscale('log')
pyplot.yscale('log')
pyplot.axvline(sls.porb.to(units.year).value)

In [ ]:
from src.selflensing.calc import LENS_MASS_DIST
mass_bin_width = LENS_MASS_DIST[0, 1] - LENS_MASS_DIST[0, 0]
pyplot.plot(LENS_MASS_DIST[0,:], LENS_MASS_DIST[1,:] / mass_bin_width)
pyplot.axvline(sls.Macc.to(units.M_sun).value)

In [ ]:
sls_fit, sampler = self_lensing_system_from_fit(
    rv_times=rvg.times,
    observed_rv=rvg.rv,
    observed_rv_err=rvg.rv/10,
    lc_times=lcg.times,
    observed_lc=lcg.flux_array,
    observed_lc_err=1 * lcg.units,
    peak_time=peak_time,
    Mcomp=sls.Mcomp,
    Rcomp=sls.Rcomp,
    Teff=sls.Teff,
    tau_eff=sls.tau_eff,
    # period_min=350 * units.day,
    # detect_periods=True,
    nsteps=5000,
    # nwalkers=20,
    burn=10,
    lensing_method=LENSING_METHOD,
)
lcg_fit = LightcurveGenerator(sls_fit, start_time=start_time, end_time=end_time, peak_time=peak_time, lensing_method=LENSING_METHOD)
rvg_fit = RVGenerator(sls_fit, start_time=start_time, end_time=end_time, peak_time=peak_time)

In [ ]:
sls_fit.a, sls_fit.Macc

In [ ]:
pyplot.scatter(lcg.times.jd, lcg.flux_array)
pyplot.plot(lcg_fit.times.jd, lcg_fit.flux_array, c='red')
pyplot.axvline(peak_time.jd)

In [ ]:
pyplot.scatter(rvg.times.jd, rvg.rv)
pyplot.plot(rvg_fit.times.jd, rvg_fit.rv, c='red')

In [ ]:
truths=[sls.a.value, sls.i.value, sls.Macc.value, sls.e.value, sls.periapsis_phase.value]

from matplotlib import pyplot as plt
labels = ["a", "i", "Macc", 'e', 'periapsis_phase']
fig, axes = plt.subplots(len(labels), figsize=(10, 7), sharex=True)
samples = sampler.get_chain()#discard=100)
flat_samples = sampler.get_chain(flat=True)
for i in range(len(labels)):
    ax = axes[i]
    ax.plot(samples[:, :, i], "k", alpha=0.3)
    ax.axhline(truths[i])
    ax.set_xlim(0, len(samples))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")

In [ ]:
import corner

fig = corner.corner(
    flat_samples, labels=labels, truths=truths
)